# Drive-backed linked resources CUJ — 2026-08-18

**Branch:** `codex/drive-backed-linked-resources`  
**Draft PR:** https://github.com/runmedev/web/pull/326

## Outcome

The implementation builds and its complete affected automated test matrix passes. The model, Drive upload/download behavior, resource-cell non-execution, cache isolation/repair/eviction, safe renderer selection, Markdown sidecar, UI states, and automation bridge all have passing coverage.

The interactive browser portion is **blocked**, so this run does not claim end-to-end playback success. The in-app Browser can open the hosted CUJ notebook at `web.runme.dev`, but that deployment does not contain this branch and reported Google Drive sync failure. The same Browser runtime cannot reach the worktree development server at `http://localhost:5173` (`ERR_CONNECTION_REFUSED`). Browser policy for this run prohibited switching to a different browser surface.

## Verification matrix

| Area | Result | Evidence |
| --- | --- | --- |
| Resource schema, Drive URL normalization, safe MIME selection | PASS | `linkedResource.test.ts` |
| Existing Drive/HTTPS attachment and local upload transaction | PASS | `linkedResourceAttachments.test.ts`, `driveResource.test.ts` |
| Resumable upload, auth retry, permission/error mapping | PASS | `drive.test.ts`, `driveResource.test.ts` |
| Principal/version cache isolation, commit marker repair, quota/LRU, memory fallback | PASS | `linkedResourceCache.test.ts` |
| Image/video/audio rendering states and object URL cleanup | PASS (component) | `ResourceCell.test.tsx` |
| Resource cells excluded from execution paths | PASS | `notebookData.test.ts`, `runmeConsole.test.ts` |
| Markdown sidecar and notebook diff integration | PASS | `serializeNotebookToMarkdown.test.ts` plus production build |
| File picker, Drive picker, drag/drop, clear-cache UI | PASS (component/build) | `Actions.test.tsx`, `DriveSyncStatusTab.test.tsx` |
| `notebooks.attach` App Console/sandbox helper | PASS | `appJsGlobals.test.ts`, `sandboxJsKernel.test.ts`, `aiAgentInstructions.test.ts` |
| Private WebM/GIF/audio playback in this branch | BLOCKED | In-app Browser cannot reach worktree server |
| Unauthorized identity and cross-principal browser cache behavior | BLOCKED | Requires private Drive fixture plus branch-accessible Browser |
| Phase 4 telemetry | NOT IMPLEMENTED | Design calls for privacy-safe metrics; no established metrics sink was added in this PR |

### Automated results

- `pnpm -C app exec vitest run <14 affected suites>`: **14 files, 252 tests passed**.
- `runme run build test`: **passed** (production build and repository test task; 7/7 react-console tests).
- `pnpm exec prettier --check <review files>`: **passed**.
- `git diff --check`: **passed**.
- Repository lint could not be run because the locked workspace does not install an `eslint` binary.

## Review findings and fixes

1. **Cache budget accounting used unrelated origin storage.** The quota-fraction budget was calculated from total browser-origin usage, so unrelated IndexedDB/OPFS data could evict linked media prematurely. Fixed by calculating the linked-media budget from committed linked-resource records while retaining the absolute origin-quota guard. Added regression coverage.
2. **Upload followed by notebook persistence failure could leave a misleading in-memory cell.** Fixed by removing the unsaved cell best-effort and returning a structured conflict error that preserves the uploaded Drive URL for recovery. Added regression coverage.
3. **The new automation method was missing from agent guidance and sandbox coverage.** Added `notebooks.attach` guidance and an end-to-end sandbox bridge test.

No GitHub review comments were present at the time of this run.

## Remaining browser CUJ

Once the branch is available to the in-app Browser (preview deployment or a Browser-reachable dev URL), repeat with the private fixture set from the design: WebM, animated GIF, audio, PDF, and unsupported HTML. Verify playback/seek, access-denied behavior, cross-principal cache isolation, no public permission creation, link-card fallback for active content, and that deleting a cell retains the Drive file.

This is the only blocker to declaring the feature fully working end to end. Automated verification found no remaining functional regressions.